# Explainable AI Framework for COVID-19 and Pneumonia Detection in Chest X-Rays
CLAHE → DenseNet121 (two-stage transfer learning) → Grad-CAM → Streamlit triage dashboard · V7 Labs COVID-19 X-ray dataset

**Before running:** Runtime ▸ Change runtime type ▸ **T4 GPU**, then Runtime ▸ **Run all**.

Proposed model ≈ 10 min on a T4; the four benchmark runs add ≈ 40 min (set `RUN_BENCHMARKS = False` in Cell 3 to skip). All numbers for the paper are written to `outputs/results_for_paper.md`.

## Cell 1 — Runtime check & installs

In [ ]:
import subprocess, sys
gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
print(gpu if gpu else "WARNING: no GPU found -> Runtime > Change runtime type > T4 GPU")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit", "kagglehub"], check=True)
print("Installs done.")

## Cell 2 — Write the shared core module (xai_core.py: CLAHE, models, Grad-CAM)

In [ ]:
from pathlib import Path
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()

CORE = r'''
"""xai_core.py - code shared by the notebook and the Streamlit dashboard.
Preprocessing (resize + CLAHE), model builders, Grad-CAM and heatmap overlays."""
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models

CLASSES = ["Normal", "Pneumonia", "COVID-19"]
IMG_SIZE = 224
CLAHE_CLIP = 2.0          # clip limit (Section IV-A)
CLAHE_TILE = (8, 8)       # 8x8 tiles
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)   # ImageNet statistics
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
ARCH_NAMES = {"densenet121": "DenseNet121", "resnet50": "ResNet50",
              "vgg16": "VGG16", "efficientnet_b0": "EfficientNet-B0"}


# ---------------------------------------------------------------- preprocessing
def read_gray(src):
    """Read an image (file path or raw bytes) as 8-bit grayscale."""
    if isinstance(src, (bytes, bytearray)):
        img = cv2.imdecode(np.frombuffer(src, np.uint8), cv2.IMREAD_UNCHANGED)
    else:
        img = cv2.imread(str(src), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError("could not decode image")
    if img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY)
        elif img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            img = img[..., 0]
    if img.dtype != np.uint8:      # e.g. 16-bit PNG
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return img


def resize224(gray):
    return cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)


def apply_clahe(gray):
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
    return clahe.apply(gray)


def preprocess(gray, use_clahe=True):
    """Resize to 224x224 (area interpolation), then CLAHE."""
    g = resize224(gray)
    return apply_clahe(g) if use_clahe else g


def to_tensor(img224):
    """uint8 224x224 -> normalised 3x224x224 float tensor (channel replicated)."""
    x = img224.astype(np.float32) / 255.0
    x = np.repeat(x[None], 3, axis=0)
    x = (x - MEAN[:, None, None]) / STD[:, None, None]
    return torch.from_numpy(x)


# ---------------------------------------------------------------- models
def build_model(arch, num_classes=3, pretrained=True, dropout=0.3):
    """ImageNet backbone; only the final classifier is replaced by Dropout + 3-way Linear.
    Returns (model, head) where head is the new classification layer."""
    def head_for(in_f):
        return nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    if arch == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = head_for(m.classifier.in_features); head = m.classifier
    elif arch == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        m.fc = head_for(m.fc.in_features); head = m.fc
    elif arch == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier[6] = head_for(m.classifier[6].in_features); head = m.classifier[6]
    elif arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = head_for(m.classifier[1].in_features); head = m.classifier
    else:
        raise ValueError(f"unknown architecture {arch}")
    return m, head


def set_backbone_trainable(model, head, trainable):
    for p in model.parameters():
        p.requires_grad = trainable
    for p in head.parameters():
        p.requires_grad = True


def get_target_layer(model, arch):
    """Last convolutional block used for Grad-CAM (7x7 maps at 224x224 input)."""
    if arch == "densenet121":
        return model.features.denseblock4      # last dense block
    if arch == "resnet50":
        return model.layer4
    if arch == "vgg16":
        return model.features[29]              # last conv + ReLU (14x14)
    if arch == "efficientnet_b0":
        return model.features[-1]
    raise ValueError(arch)


def load_model(path, device="cpu"):
    ckpt = torch.load(path, map_location=device)
    model, _ = build_model(ckpt["arch"], len(ckpt["classes"]), pretrained=False,
                           dropout=ckpt.get("dropout", 0.3))
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()
    meta = {k: v for k, v in ckpt.items() if k != "state_dict"}
    return model, meta


# ---------------------------------------------------------------- Grad-CAM
class GradCAM:
    """alpha_k^c = mean_ij dy^c/dA^k_ij ;  L^c = ReLU(sum_k alpha_k^c A^k)   (Eq. 6)"""
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.handle = layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inp, out):
        self.acts = out
        if out.requires_grad:
            out.register_hook(self._save_grad)

    def _save_grad(self, grad):
        self.grads = grad

    def __call__(self, x, class_idx=None):
        """x: 1x3xHxW tensor. Returns (cam in [0,1] at feature-map size, probs, class)."""
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        x = x.clone().requires_grad_(True)
        with torch.enable_grad():
            logits = self.model(x)
            probs = torch.softmax(logits.float(), dim=1)[0]
            c = int(probs.argmax()) if class_idx is None else int(class_idx)
            logits[0, c].backward()
        A, G = self.acts.detach()[0].float(), self.grads.detach()[0].float()
        weights = G.mean(dim=(1, 2))
        cam = torch.relu((weights[:, None, None] * A).sum(0))
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy(), probs.detach().cpu().numpy(), c

    def remove(self):
        self.handle.remove()


def upsample_cam(cam, size=IMG_SIZE):
    return cv2.resize(cam.astype(np.float32), (size, size), interpolation=cv2.INTER_LINEAR)


def overlay(img224, cam, alpha=0.4):
    """I_o = alpha * colormap(L) + (1 - alpha) * I   (alpha = 0.4)"""
    h, w = img224.shape[:2]
    heat = cv2.resize(cam.astype(np.float32), (w, h), interpolation=cv2.INTER_LINEAR)
    heat = cv2.applyColorMap(np.uint8(255 * np.clip(heat, 0, 1)), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    base = cv2.cvtColor(img224, cv2.COLOR_GRAY2RGB) if img224.ndim == 2 else img224
    return (alpha * heat + (1 - alpha) * base).clip(0, 255).astype(np.uint8)


def upscale(img, size=448):
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_CUBIC)


def explain(model, arch, gray_raw, use_clahe=True, device="cpu", alpha=0.4):
    """Full inference: preprocess -> predict -> Grad-CAM -> overlay."""
    img = preprocess(gray_raw, use_clahe)
    x = to_tensor(img)[None].to(device)
    gc = GradCAM(model, get_target_layer(model, arch))
    try:
        cam, probs, c = gc(x)
    finally:
        gc.remove()
    return {"img": img, "probs": probs, "pred": c, "cam": cam, "overlay": overlay(img, cam, alpha)}
'''
(ROOT / "xai_core.py").write_text(CORE.lstrip())
print("Wrote", ROOT / "xai_core.py")

## Cell 3 — Imports & configuration (Table II hyperparameters)

In [ ]:
import os, re, sys, json, time, random, shutil, zipfile, subprocess, importlib, gc
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report, accuracy_score)

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
sys.path.insert(0, str(ROOT))
import xai_core; importlib.reload(xai_core)
from xai_core import *

# ---- hyperparameters (Table II) ----
SEED           = 42
BATCH_SIZE     = 32
EPOCHS         = 12      # total epochs
WARMUP_EPOCHS  = 2       # frozen-backbone epochs (head only)
LR_HEAD        = 1e-3
LR_FINETUNE    = 1e-4
DROPOUT        = 0.3
CONF_THRESHOLD = 0.70    # dashboard review flag
NUM_WORKERS    = 2

RUN_BENCHMARKS = True    # ResNet50, VGG16, EfficientNet-B0, no-CLAHE ablation (~40 min extra on T4)
SAVE_TO_DRIVE  = False   # True = also copy outputs to Google Drive (survives runtime resets)

WORK, OUT = ROOT / "work", ROOT / "outputs"
FIG, MODELS, SAMPLES = OUT / "figures", OUT / "models", OUT / "samples"
for d in (WORK, OUT, FIG, MODELS, SAMPLES):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
print(f"Device: {DEVICE_NAME} | torch {torch.__version__}")

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"command failed: {cmd}")
    return r.stdout

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight", "font.size": 10})

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/xai_cxr"); DRIVE_DIR.mkdir(parents=True, exist_ok=True)

## Cell 4 — V7 annotations -> labels, exclusions, lung polygons (Section III)

In [ ]:
V7_DIR, ANN_DIR = WORK / "v7", WORK / "v7_json"
if not (V7_DIR / "annotations" / "all-images.zip").exists():
    shutil.rmtree(V7_DIR, ignore_errors=True)
    sh(f"git clone -q --depth 1 --filter=blob:none --sparse https://github.com/v7labs/covid-19-xray-dataset.git {V7_DIR}")
    sh("git sparse-checkout set --no-cone /annotations/all-images.zip /README.md", cwd=V7_DIR)
if not ANN_DIR.exists():
    with zipfile.ZipFile(V7_DIR / "annotations" / "all-images.zip") as z:
        z.extractall(ANN_DIR)
json_files = sorted(ANN_DIR.rglob("*.json"))
print(f"{len(json_files)} annotation files")

HEALTHY = "No Pneumonia (healthy)"
EXCLUDE_TAGS = {"CT", "View/Axial", "View/Lateral", "View/Coronal"}

def map_label(tags):
    """COVID-19 takes priority; then any pneumonia tag; then healthy (Table I-b)."""
    if "COVID-19" in tags:
        return "COVID-19"
    if any("pneumonia" in t.lower() and t != HEALTHY for t in tags):
        return "Pneumonia"
    if HEALTHY in tags:
        return "Normal"
    return None

def patient_id(fname, tags):
    for t in tags:
        if t.lower().startswith("patientid:"):
            return "ieee-" + t.split(":", 1)[1]
    m = re.match(r"^(person\d+)_", fname) or re.match(r"^((?:NORMAL2-)?IM-\d+)-", fname)
    return m.group(1) if m else None

records, LUNG_POLYS = [], {}
for jf in json_files:
    d = json.load(open(jf))
    im = d["image"]
    tags = {a["name"].strip() for a in d["annotations"] if "tag" in a}
    names = {a["name"] for a in d["annotations"]}
    label = map_label(tags)
    if tags & EXCLUDE_TAGS:
        reason = "CT / non-frontal view"
    elif "Ignore" in names:
        reason = "Ignore region"
    elif label is None:
        reason = "no diagnosis tag"
    else:
        reason = ""
    polys = []
    for a in d["annotations"]:
        if a["name"] != "Lung":
            continue
        paths = [a["polygon"]["path"]] if "polygon" in a else a.get("complex_polygon", {}).get("path", [])
        for p in paths:
            polys.append(np.array([[pt["x"] / im["width"], pt["y"] / im["height"]] for pt in p], np.float32))
    LUNG_POLYS[jf.stem] = polys
    records.append(dict(json_id=jf.stem, orig_name=im["original_filename"], url=im.get("url", ""),
                        ann_w=im["width"], ann_h=im["height"], label=label, excluded=reason,
                        patient=patient_id(im["original_filename"], tags), tags="|".join(sorted(tags))))

df_all = pd.DataFrame(records)
print("\nExcluded images:"); print(df_all[df_all.excluded != ""].excluded.value_counts().to_string())
df = df_all[df_all.excluded == ""].reset_index(drop=True)
print("\nKept images per class:"); print(df.label.value_counts().reindex(CLASSES).to_string())

## Cell 5 — Fetch the images

In [ ]:
# The V7 annotations hold signed Darwin links from 2020. If they still work we use them;
# otherwise the identical original images are taken from their public sources:
#   - COVID-19 / other web images  -> ieee8023/covid-chestxray-dataset (GitHub)
#   - Normal / pediatric pneumonia -> Kermany et al. chest-xray-pneumonia (Kaggle)
# Labels and exclusions always come from the V7 annotations (Cell 4).
import requests
IMG_DIR = WORK / "v7_images"; IMG_DIR.mkdir(exist_ok=True)

def fetch_v7(row):
    dst = IMG_DIR / f"{row.json_id}{Path(row.orig_name).suffix or '.png'}"
    if dst.exists() and dst.stat().st_size > 0:
        return str(dst)
    try:
        r = requests.get(row.url, timeout=10)
        if r.status_code == 200 and r.headers.get("content-type", "").startswith("image"):
            dst.write_bytes(r.content); return str(dst)
    except Exception:
        pass
    return None

probe = [fetch_v7(r) for r in df.sample(6, random_state=SEED).itertuples()]
if sum(p is not None for p in probe) >= 5:
    print("V7 Darwin links are live -> downloading all images from V7 ...")
    with ThreadPoolExecutor(16) as ex:
        df["src_path"] = list(ex.map(fetch_v7, df.itertuples()))
else:
    print("V7 Darwin links have expired -> using the original public sources of the same images.")
    IEEE_DIR = WORK / "ieee8023"
    if not (IEEE_DIR / "images").exists():
        shutil.rmtree(IEEE_DIR, ignore_errors=True)
        sh(f"git clone -q --depth 1 --filter=blob:none --sparse https://github.com/ieee8023/covid-chestxray-dataset.git {IEEE_DIR}")
        sh("git sparse-checkout set images", cwd=IEEE_DIR)
    import kagglehub
    try:
        KERMANY_DIR = Path(kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia"))
    except Exception as e:
        raise RuntimeError("Kaggle download failed. Add your Kaggle credentials: Colab key icon (Secrets) -> "
                           "KAGGLE_USERNAME and KAGGLE_KEY (from kaggle.com > Settings > API), then rerun.") from e
    index = {}
    for base in (IEEE_DIR / "images", KERMANY_DIR):
        for dirpath, _, files in os.walk(base):
            if "__MACOSX" in dirpath:
                continue
            for f in files:
                index.setdefault(f, os.path.join(dirpath, f))
    df["src_path"] = df.orig_name.map(index)

missing = df.src_path.isna()
print(f"\nImages found: {(~missing).sum()} / {len(df)}")
if missing.any():
    print("Not found (dropped):"); print(df[missing].label.value_counts().to_string())
df = df[~missing].reset_index(drop=True)

## Cell 6 — Read, resize (224x224, area) and CLAHE-enhance every image

In [ ]:
CACHE = WORK / "cache_224.npz"

def load_one(path):
    try:
        g = read_gray(path)
        return resize224(g), g.shape
    except Exception:
        return None, None

if CACHE.exists():
    c = np.load(CACHE, allow_pickle=True)
    df = pd.read_csv(WORK / "records.csv", dtype={"json_id": str})
    X_RAW, X_CLAHE = c["raw"], c["clahe"]
    print("Loaded cached arrays")
else:
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        out = list(ex.map(load_one, df.src_path))
    ok = np.array([o[0] is not None for o in out])
    print(f"Unreadable files dropped: {(~ok).sum()}")
    df = df[ok].reset_index(drop=True)
    out = [o for o, k in zip(out, ok) if k]
    df["orig_h"] = [o[1][0] for o in out]; df["orig_w"] = [o[1][1] for o in out]
    X_RAW = np.stack([o[0] for o in out])
    X_CLAHE = np.stack([apply_clahe(x) for x in X_RAW])
    np.savez(CACHE, raw=X_RAW, clahe=X_CLAHE)
    df.to_csv(WORK / "records.csv", index=False)
    print(f"Preprocessed {len(df)} images in {time.time()-t0:.0f}s")

y = df.label.map(CLASSES.index).values
print("Final class counts:", dict(zip(CLASSES, np.bincount(y, minlength=3))))

## Cell 7 — Stratified 70/15/15 split (seed 42) -> Table I, Fig. 2, dashboard samples

In [ ]:
idx_all = np.arange(len(y))
idx_train, idx_tmp = train_test_split(idx_all, test_size=0.30, stratify=y, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.50, stratify=y[idx_tmp], random_state=SEED)
df["split"] = "train"; df.loc[idx_val, "split"] = "val"; df.loc[idx_test, "split"] = "test"
df.to_csv(OUT / "dataset_split.csv", index=False)

table1 = pd.crosstab(df.split, df.label).reindex(index=["train", "val", "test"], columns=CLASSES)
table1["Images"] = table1.sum(1)
table1["Share"] = (100 * table1.Images / len(df)).round(1).astype(str) + "%"
table1.index = pd.Index(["Train", "Validation", "Test"], name="Subset"); table1.columns.name = None
print("TABLE I - DATASET SPLIT"); display(table1)
table1.to_csv(OUT / "table1_split.csv")

# Fig. 2: one test radiograph before/after CLAHE with histograms
i_demo = idx_test[y[idx_test] == CLASSES.index("COVID-19")][0]
fig, ax = plt.subplots(2, 2, figsize=(8, 7), gridspec_kw={"height_ratios": [3, 1.3]})
for j, (img, title) in enumerate([(X_RAW[i_demo], "Original"), (X_CLAHE[i_demo], "CLAHE (clip 2.0, 8x8)")]):
    ax[0, j].imshow(img, cmap="gray", vmin=0, vmax=255); ax[0, j].set_title(title); ax[0, j].axis("off")
    ax[1, j].hist(img.ravel(), bins=256, range=(0, 255), color="#4C72B0")
    ax[1, j].set_xlim(0, 255); ax[1, j].set_xlabel("Intensity"); ax[1, j].set_yticks([])
plt.tight_layout(); plt.savefig(FIG / "clahe_example.png"); plt.show()

# copy a few original test images for the dashboard's "sample" picker
for c in CLASSES:
    for k, i in enumerate(idx_test[y[idx_test] == CLASSES.index(c)][:3]):
        src = Path(df.src_path[i])
        shutil.copy(src, SAMPLES / f"{c.replace('-', '')}_{k+1}{src.suffix.lower()}")

## Cell 8 — Dataset, training loop, metrics and plotting utilities

In [ ]:
class CXRDataset(Dataset):
    """Rotation <=10 deg + mild random-resized crop; no horizontal flips (Section IV-D)."""
    def __init__(self, images, labels, train=False):
        self.images, self.labels = images, labels
        self.aug = T.Compose([T.RandomRotation(10),
                              T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05), antialias=True)
                              ]) if train else None
        self.mean = torch.tensor(MEAN).view(3, 1, 1); self.std = torch.tensor(STD).view(3, 1, 1)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        x = torch.from_numpy(self.images[i]).float().div(255).unsqueeze(0)
        if self.aug is not None:
            x = self.aug(x)
        x = (x.repeat(3, 1, 1) - self.mean) / self.std
        return x, int(self.labels[i])

def make_loaders(X):
    y_tr = y[idx_train]
    counts = np.bincount(y_tr, minlength=len(CLASSES))
    sampler = WeightedRandomSampler(torch.as_tensor(1.0 / counts[y_tr], dtype=torch.double),
                                    num_samples=len(y_tr), replacement=True,
                                    generator=torch.Generator().manual_seed(SEED))      # class-balanced batches
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    train_dl = DataLoader(CXRDataset(X[idx_train], y_tr, train=True), sampler=sampler,
                          generator=torch.Generator().manual_seed(SEED), **kw)
    val_dl = DataLoader(CXRDataset(X[idx_val], y[idx_val]), shuffle=False, **kw)
    test_dl = DataLoader(CXRDataset(X[idx_test], y[idx_test]), shuffle=False, **kw)
    class_w = len(y_tr) / (len(CLASSES) * counts)                                      # w_c = N / (C * N_c)  (Eq. 5)
    return train_dl, val_dl, test_dl, class_w

@torch.no_grad()
def run_eval(model, dl, criterion):
    model.eval(); probs, trues, tot, n = [], [], 0.0, 0
    for xb, yb in dl:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            out = model(xb)
        out = out.float()
        tot += criterion(out, yb).item() * len(yb); n += len(yb)
        probs.append(torch.softmax(out, 1).cpu().numpy()); trues.append(yb.cpu().numpy())
    return tot / n, np.concatenate(probs), np.concatenate(trues)

def compute_metrics(true, prob):
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(true, pred, average="macro", zero_division=0)
    pc, rc, fc, sc = precision_recall_fscore_support(true, pred, labels=list(range(len(CLASSES))), zero_division=0)
    return {"accuracy": accuracy_score(true, pred), "precision": p, "recall": r, "f1": f,
            "auc": roc_auc_score(true, prob, multi_class="ovr", average="macro"),
            "per_class": {c: {"precision": pc[k], "recall": rc[k], "f1": fc[k], "support": int(sc[k])}
                          for k, c in enumerate(CLASSES)},
            "confusion": confusion_matrix(true, pred, labels=list(range(len(CLASSES)))).tolist()}

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def train_model(arch, use_clahe, tag):
    """Algorithm 1: two-stage transfer learning, best-validation-accuracy checkpoint, single test evaluation."""
    print(f"\n=== {tag}: {ARCH_NAMES[arch]} | CLAHE={use_clahe} ===")
    seed_everything()
    train_dl, val_dl, test_dl, class_w = make_loaders(X_CLAHE if use_clahe else X_RAW)
    model, head = build_model(arch, len(CLASSES), pretrained=True, dropout=DROPOUT)
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_w, dtype=torch.float32, device=DEVICE))
    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    hist, best_acc, best_epoch, best_state = [], -1.0, 0, None
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        if epoch == 1 or epoch == WARMUP_EPOCHS + 1:
            frozen = epoch <= WARMUP_EPOCHS
            set_backbone_trainable(model, head, trainable=not frozen)
            opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                                   lr=LR_HEAD if frozen else LR_FINETUNE)
        model.train(); tl, tc, n = 0.0, 0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                out = model(xb)
                loss = criterion(out.float(), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(yb); tc += (out.argmax(1) == yb).sum().item(); n += len(yb)
        vl, vprob, vtrue = run_eval(model, val_dl, criterion)
        va = float((vprob.argmax(1) == vtrue).mean())
        hist.append(dict(epoch=epoch, phase="head" if frozen else "finetune",
                         train_loss=tl / n, train_acc=tc / n, val_loss=vl, val_acc=va))
        star = ""
        if va > best_acc:
            best_acc, best_epoch, star = va, epoch, "  <- best"
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"ep {epoch:2d} [{'head' if frozen else 'full'}] train loss {tl/n:.4f} acc {tc/n:.4f} | "
              f"val loss {vl:.4f} acc {va:.4f} | {(time.time()-t0)/60:.1f} min{star}")
    train_min = (time.time() - t0) / 60
    model.load_state_dict(best_state)
    _, prob, true = run_eval(model, test_dl, criterion)       # test set evaluated once
    m = compute_metrics(true, prob)
    print(f"TEST  acc {m['accuracy']:.4f} | macro P {m['precision']:.4f} R {m['recall']:.4f} "
          f"F1 {m['f1']:.4f} AUC {m['auc']:.4f} | best epoch {best_epoch} | {train_min:.1f} min")
    torch.save({"arch": arch, "use_clahe": use_clahe, "classes": CLASSES, "dropout": DROPOUT,
                "best_val_acc": best_acc, "best_epoch": best_epoch, "state_dict": model.state_dict()},
               MODELS / f"{tag}.pt")
    pd.DataFrame(hist).to_csv(OUT / f"history_{tag}.csv", index=False)
    json.dump({**m, "train_minutes": train_min, "best_epoch": best_epoch, "best_val_acc": best_acc},
              open(OUT / f"metrics_{tag}.json", "w"), indent=2, default=float)
    return dict(tag=tag, arch=arch, use_clahe=use_clahe, history=hist, metrics=m, y_true=true, y_prob=prob,
                train_min=train_min, params=count_params(model), model=model)

def plot_curves(hist, path, title=""):
    h = pd.DataFrame(hist)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    for a, key, lab in [(ax[0], "loss", "Loss"), (ax[1], "acc", "Accuracy")]:
        a.plot(h.epoch, h[f"train_{key}"], "o-", label="Train")
        a.plot(h.epoch, h[f"val_{key}"], "s-", label="Validation")
        a.axvline(WARMUP_EPOCHS + 0.5, ls="--", c="gray", lw=1)
        a.text(WARMUP_EPOCHS + 0.6, a.get_ylim()[1], " unfreeze", va="top", color="gray", fontsize=8)
        a.set_xlabel("Epoch"); a.set_ylabel(lab); a.grid(alpha=.3); a.legend()
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path); plt.show()

def plot_confusion(cm, path, title=""):
    cm = np.array(cm); cmn = cm / cm.sum(1, keepdims=True)
    fig, ax = plt.subplots(figsize=(5, 4.3))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            ax.text(j, i, f"{cm[i, j]}\n({100*cmn[i, j]:.1f}%)", ha="center", va="center",
                    color="white" if cmn[i, j] > .5 else "black")
    ax.set_xticks(range(3), CLASSES); ax.set_yticks(range(3), CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.savefig(path); plt.show()

def plot_roc(true, prob, path, title=""):
    fig, ax = plt.subplots(figsize=(5, 4.5))
    for k, c in enumerate(CLASSES):
        fpr, tpr, _ = roc_curve(true == k, prob[:, k])
        ax.plot(fpr, tpr, lw=2, label=f"{c} (AUC = {roc_auc_score(true == k, prob[:, k]):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.set_title(title)
    ax.legend(loc="lower right"); ax.grid(alpha=.3); plt.tight_layout(); plt.savefig(path); plt.show()

print("Utilities ready.")

## Cell 9 — Train the proposed model (DenseNet121 + CLAHE) -> Figs. 3-5, Table IV

In [ ]:
results = {}
res = train_model("densenet121", use_clahe=True, tag="densenet121_clahe")
proposed_model = res.pop("model")
results[res["tag"]] = res

plot_curves(res["history"], FIG / "curves.png", "DenseNet121 + CLAHE")
plot_confusion(res["metrics"]["confusion"], FIG / "confusion.png", "Test confusion matrix")
plot_roc(res["y_true"], res["y_prob"], FIG / "roc.png", "One-vs-rest ROC (test)")

table4 = pd.DataFrame(res["metrics"]["per_class"]).T[["precision", "recall", "f1", "support"]]
table4[["precision", "recall", "f1"]] = (100 * table4[["precision", "recall", "f1"]].astype(float)).round(2)
table4["support"] = table4["support"].astype(int); table4.index.name = "Class"
print("TABLE IV - PER-CLASS RESULTS (proposed)"); display(table4)
table4.to_csv(OUT / "table4_per_class.csv")
print(classification_report(res["y_true"], res["y_prob"].argmax(1), target_names=CLASSES, digits=4))

## Cell 10 — Grad-CAM gallery (Fig. 6), lung-focus check and error analysis

In [ ]:
proposed_model.eval()
gcam = GradCAM(proposed_model, get_target_layer(proposed_model, "densenet121"))
CAMS = []
for i in idx_test:                                   # same order as the test loader
    cam, _, _ = gcam(to_tensor(X_CLAHE[i])[None].to(DEVICE))
    CAMS.append(cam)
gcam.remove()
CAMS = np.stack(CAMS)
t_true, t_prob = res["y_true"], res["y_prob"]
t_pred, t_conf = t_prob.argmax(1), t_prob.max(1)
rng = np.random.default_rng(SEED)

# --- Fig. 6: correctly classified images, chosen at random (not hand-picked)
N_PER_CLASS = 4
fig, axes = plt.subplots(3, 2 * N_PER_CLASS, figsize=(2 * 2 * N_PER_CLASS, 7))
for r, c in enumerate(CLASSES):
    pool = np.where((t_true == r) & (t_pred == r))[0]
    pick = rng.choice(pool, size=min(N_PER_CLASS, len(pool)), replace=False)
    for k in range(N_PER_CLASS):
        a1, a2 = axes[r, 2 * k], axes[r, 2 * k + 1]
        a1.axis("off"); a2.axis("off")
        if k >= len(pick):
            continue
        j = pick[k]; i = idx_test[j]
        a1.imshow(X_CLAHE[i], cmap="gray"); a2.imshow(overlay(X_CLAHE[i], CAMS[j]))
        a2.set_title(f"p = {t_conf[j]:.2f}", fontsize=9)
    axes[r, 0].text(-0.15, 0.5, c, transform=axes[r, 0].transAxes, rotation=90, va="center", ha="right",
                    fontsize=12, fontweight="bold")
plt.suptitle("Correctly classified test images (CLAHE) and Grad-CAM overlays", y=1.0)
plt.tight_layout(); plt.savefig(FIG / "gradcam_gallery.png"); plt.show()

# --- quantitative plausibility check: share of Grad-CAM energy inside the V7 lung polygons
def lung_mask(json_id):
    m = np.zeros((IMG_SIZE, IMG_SIZE), np.uint8)
    for p in LUNG_POLYS.get(str(json_id), []):
        cv2.fillPoly(m, [np.round(p * IMG_SIZE).astype(np.int32)], 1)
    return m

lung_rows = []
for j, i in enumerate(idx_test):
    m = lung_mask(df.json_id[i])
    if m.sum() == 0:
        continue
    cam = upsample_cam(CAMS[j])
    lung_rows.append(dict(cls=CLASSES[t_true[j]], correct=bool(t_true[j] == t_pred[j]),
                          energy_in_lung=float((cam * m).sum() / (cam.sum() + 1e-8)), lung_area=float(m.mean())))
lung_df = pd.DataFrame(lung_rows)
lung_summary = lung_df.groupby("cls")[["energy_in_lung", "lung_area"]].mean().reindex(CLASSES)
lung_summary.loc["All"] = lung_df[["energy_in_lung", "lung_area"]].mean()
lung_summary.index.name = "Class"
print("Grad-CAM energy inside lung polygons (vs. lung area = value expected from a uniform map):")
display((100 * lung_summary).round(1))

# --- error analysis
wrong = np.where(t_true != t_pred)[0]
pairs = Counter((CLASSES[t_true[j]], CLASSES[t_pred[j]]) for j in wrong)
print(f"\nMisclassified: {len(wrong)} / {len(t_true)}")
for (a, b), n in pairs.most_common():
    print(f"  true {a:9s} -> predicted {b:9s}: {n}")
ok_mask = t_true == t_pred
err_stats = pd.DataFrame({
    "mean confidence": [t_conf[ok_mask].mean(), t_conf[~ok_mask].mean() if len(wrong) else np.nan],
    "median original short side (px)": [np.median(np.minimum(df.orig_h, df.orig_w).values[idx_test][ok_mask]),
                                        np.median(np.minimum(df.orig_h, df.orig_w).values[idx_test][~ok_mask]) if len(wrong) else np.nan],
    "mean raw contrast (pixel std)": [X_RAW[idx_test][ok_mask].std(axis=(1, 2)).mean(),
                                      X_RAW[idx_test][~ok_mask].std(axis=(1, 2)).mean() if len(wrong) else np.nan],
    "Grad-CAM energy in lungs (%)": [100 * lung_df[lung_df.correct].energy_in_lung.mean(),
                                     100 * lung_df[~lung_df.correct].energy_in_lung.mean()],
}, index=["correct", "misclassified"]).round(3)
display(err_stats); err_stats.to_csv(OUT / "error_analysis.csv")

if len(wrong):
    show = rng.choice(wrong, size=min(8, len(wrong)), replace=False)
    fig, axes = plt.subplots(2, len(show), figsize=(2.3 * len(show), 5.2), squeeze=False)
    for k, j in enumerate(show):
        i = idx_test[j]
        axes[0, k].imshow(X_CLAHE[i], cmap="gray"); axes[1, k].imshow(overlay(X_CLAHE[i], CAMS[j]))
        axes[0, k].set_title(f"T: {CLASSES[t_true[j]]}\nP: {CLASSES[t_pred[j]]} ({t_conf[j]:.2f})", fontsize=8)
        axes[0, k].axis("off"); axes[1, k].axis("off")
    plt.suptitle("Misclassified test images (random sample)"); plt.tight_layout()
    plt.savefig(FIG / "errors.png"); plt.show()

## Cell 11 — Benchmarks + CLAHE ablation (same split, same recipe) -> Table III

In [ ]:
BENCHMARKS = [("densenet121", False, "densenet121_noclahe"),
              ("resnet50", True, "resnet50"),
              ("vgg16", True, "vgg16"),
              ("efficientnet_b0", True, "efficientnet_b0")]
if RUN_BENCHMARKS:
    for arch, clahe, tag in BENCHMARKS:
        if tag in results:
            continue
        r = train_model(arch, clahe, tag)
        r.pop("model"); results[tag] = r
        plot_curves(r["history"], FIG / f"curves_{tag}.png", tag)
        gc.collect(); torch.cuda.empty_cache()
else:
    print("RUN_BENCHMARKS = False -> skipped")

## Cell 12 — Tables III & V, comparison chart, results_for_paper.md

In [ ]:
NAMES = {"densenet121_clahe": "DenseNet121+CLAHE (ours)", "densenet121_noclahe": "DenseNet121, no CLAHE",
         "resnet50": "ResNet50", "vgg16": "VGG16", "efficientnet_b0": "EfficientNet-B0"}
order = [t for t in NAMES if t in results]
table3 = pd.DataFrame([{"Model": NAMES[t],
                        **{k: round(100 * results[t]["metrics"][m], 2) for k, m in
                           [("Acc", "accuracy"), ("Prec", "precision"), ("Rec", "recall"), ("F1", "f1"), ("AUC", "auc")]},
                        "Min": round(results[t]["train_min"], 1)} for t in order]).set_index("Model")
print("TABLE III - TEST-SET COMPARISON (%)"); display(table3); table3.to_csv(OUT / "table3_comparison.csv")

table5 = pd.DataFrame([{"Model": ARCH_NAMES[a],
                        "Parameters (M)": round(count_params(build_model(a, 3, pretrained=False)[0]) / 1e6, 2)}
                       for a in ARCH_NAMES]).set_index("Model")
print("TABLE V - BACKBONE SIZE (with 3-way head)"); display(table5); table5.to_csv(OUT / "table5_params.csv")

if len(order) > 1:
    ax = table3[["Acc", "F1", "AUC"]].plot.bar(figsize=(9, 4), rot=15, width=.75)
    ax.set_ylabel("%"); ax.set_ylim(max(0, table3[["Acc", "F1", "AUC"]].values.min() - 10), 100)
    ax.grid(axis="y", alpha=.3); ax.set_title("Test-set comparison")
    plt.tight_layout(); plt.savefig(FIG / "model_comparison.png"); plt.show()

def md_table(d):
    d = d.reset_index()
    lines = ["| " + " | ".join(map(str, d.columns)) + " |", "|" + "---|" * len(d.columns)]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in d.values]
    return "\n".join(lines)

pm = results["densenet121_clahe"]["metrics"]
pct = lambda v: f"{100 * v:.2f}"
md = f"""# Results to paste into the paper  (generated {time.strftime('%Y-%m-%d %H:%M')})

## Abstract / Conclusion
- Accuracy: **{pct(pm['accuracy'])}%**  |  Macro F1: **{pct(pm['f1'])}%**  |  Macro one-vs-rest AUC: **{pct(pm['auc'])}%**

## Section V - environment
- Training device: **{DEVICE_NAME}** (PyTorch {torch.__version__}); proposed model trained in {results['densenet121_clahe']['train_min']:.1f} min
- Best validation epoch: {results['densenet121_clahe']['history'][int(np.argmax([h['val_acc'] for h in results['densenet121_clahe']['history']]))]['epoch']} of {EPOCHS}

## Table I - dataset split
{md_table(table1)}

Excluded from {len(df_all)} annotated images: {", ".join(f"{k}: {v}" for k, v in df_all[df_all.excluded != ""].excluded.value_counts().items())}.

## Table III - test-set comparison (%)
{md_table(table3)}

## Table IV - per-class results of the proposed model (%)
{md_table(table4)}

## Table V - parameters (computed, 3-way head)
{md_table(table5)}

## Grad-CAM lung-focus check (share of heatmap energy inside V7 lung polygons, %)
{md_table((100 * lung_summary).round(1))}
A uniform heatmap would put "lung_area" % of its energy in the lungs; values well above that mean the maps concentrate on lung fields.

## Error analysis facts
- Misclassified: {len(wrong)} of {len(t_true)} test images
{chr(10).join(f"- true {a} -> predicted {b}: {n}" for (a, b), n in pairs.most_common())}

{md_table(err_stats)}

## Figures (outputs/figures/)
- Fig. 2 clahe_example.png | Fig. 3 curves.png | Fig. 4 confusion.png | Fig. 5 roc.png | Fig. 6 gradcam_gallery.png
- Extra: errors.png (Section VI-G), model_comparison.png, curves_<model>.png
"""
(OUT / "results_for_paper.md").write_text(md)
print(md)

## Cell 13 — Save & download outputs

In [ ]:
zip_path = ROOT / "xai_cxr_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file() and not (f.suffix == ".pt" and f.stem != "densenet121_clahe"):   # skip large benchmark weights
            z.write(f, f.relative_to(OUT.parent))
print(f"Zipped -> {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
if SAVE_TO_DRIVE:
    shutil.copytree(OUT, DRIVE_DIR / "outputs", dirs_exist_ok=True)
    shutil.copy(ROOT / "xai_core.py", DRIVE_DIR)
    print("Copied to", DRIVE_DIR)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Download it from the Colab file browser:", e)

## Cell 14 — Write the Streamlit triage dashboard (app.py)

In [ ]:
from pathlib import Path
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
APP = r'''
import os
from pathlib import Path
import numpy as np
import streamlit as st
import torch
from xai_core import CLASSES, ARCH_NAMES, read_gray, resize224, load_model, explain, upscale

st.set_page_config(page_title="Chest X-ray XAI Triage", page_icon="🫁", layout="wide")
MODEL_PATH = os.environ.get("MODEL_PATH", "outputs/models/densenet121_clahe.pt")
SAMPLES_DIR = Path(os.environ.get("SAMPLES_DIR", "outputs/samples"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@st.cache_resource
def get_model(path):
    return load_model(path, DEVICE)

model, meta = get_model(MODEL_PATH)

st.title("🫁 Explainable COVID-19 & Pneumonia Triage")
st.caption(f"{ARCH_NAMES[meta['arch']]} + {'CLAHE + ' if meta['use_clahe'] else ''}Grad-CAM  ·  "
           "Research prototype - not a diagnostic device")

with st.sidebar:
    st.header("Settings")
    thr = st.slider("Flag for radiologist review below confidence", 0.50, 0.99, 0.70, 0.01)
    alpha = st.slider("Heatmap opacity", 0.1, 0.9, 0.4, 0.05)
    source = st.radio("Image source", ["Upload", "Sample test image"])
    st.divider()
    st.markdown(f"**Model:** {ARCH_NAMES[meta['arch']]}  \n**CLAHE:** {meta['use_clahe']}  \n"
                f"**Best val. accuracy:** {100 * meta.get('best_val_acc', float('nan')):.2f}%  \n**Device:** {DEVICE}")

img_bytes, name = None, None
if source == "Upload":
    f = st.file_uploader("Frontal (PA/AP) chest X-ray", type=["png", "jpg", "jpeg"])
    if f is not None:
        img_bytes, name = f.getvalue(), f.name
else:
    samples = sorted(p for p in SAMPLES_DIR.glob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg"))
    if samples:
        pick = st.selectbox("Sample (held-out test set; true class in file name)", samples, format_func=lambda p: p.name)
        img_bytes, name = pick.read_bytes(), pick.name
    else:
        st.warning(f"No samples found in {SAMPLES_DIR}")

if img_bytes is None:
    st.info("Upload a chest X-ray to get a prediction, confidence and Grad-CAM heatmap.")
    st.stop()

try:
    gray = read_gray(img_bytes)
except ValueError:
    st.error("Could not read this file as an image.")
    st.stop()

with st.spinner("Analysing ..."):
    r = explain(model, meta["arch"], gray, meta["use_clahe"], DEVICE, alpha=alpha)
pred, probs = CLASSES[r["pred"]], r["probs"]
conf = float(probs[r["pred"]])

c1, c2, c3 = st.columns(3)
c1.image(upscale(resize224(gray)), caption="Original (resized 224x224)")
c2.image(upscale(r["img"]), caption="CLAHE-enhanced model input" if meta["use_clahe"] else "Model input")
c3.image(upscale(r["overlay"]), caption=f"Grad-CAM evidence for '{pred}'")

st.divider()
left, right = st.columns([1, 2])
with left:
    st.metric("Predicted class", pred, f"{100 * conf:.1f}% confidence", delta_color="off")
    if conf < thr:
        st.warning(f"Confidence below {100 * thr:.0f}% - flag for radiologist review.")
    else:
        st.success(f"Confidence at or above {100 * thr:.0f}% threshold.")
with right:
    st.subheader("Class probabilities")
    for k, c in enumerate(CLASSES):
        st.progress(float(probs[k]), text=f"{c}: {100 * probs[k]:.1f}%")

with st.expander("How to read the heatmap"):
    st.markdown("Red/yellow regions contributed most to the predicted class (Grad-CAM on the last dense block, "
                "7x7 resolution, so it marks coarse regions, not individual lesions). A plausible map focuses on "
                "the lung fields; heat on borders, text markers or devices suggests a shortcut and the prediction "
                "should not be trusted. The threshold is a design choice, not a calibrated operating point.")
'''
(ROOT / "app.py").write_text(APP.lstrip())
print("Wrote", ROOT / "app.py")

## Cell 15 — Launch the dashboard (public link via Cloudflare tunnel)

In [ ]:
import os, re, sys, time, subprocess, urllib.request
from pathlib import Path
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
MODEL_FILE = ROOT / "outputs" / "models" / "densenet121_clahe.pt"
if not MODEL_FILE.exists() and Path("/content/drive/MyDrive/xai_cxr/outputs/models/densenet121_clahe.pt").exists():
    MODEL_FILE = Path("/content/drive/MyDrive/xai_cxr/outputs/models/densenet121_clahe.pt")
assert MODEL_FILE.exists(), "Train the model first (Cell 9) or upload densenet121_clahe.pt to outputs/models/"

CF = ROOT / "cloudflared"
if not CF.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    CF.chmod(0o755)
subprocess.run("pkill -f 'streamlit run'; pkill -f 'cloudflared tunnel'", shell=True)
time.sleep(1)
env = dict(os.environ, MODEL_PATH=str(MODEL_FILE), SAMPLES_DIR=str(ROOT / "outputs" / "samples"))
subprocess.Popen([sys.executable, "-m", "streamlit", "run", str(ROOT / "app.py"), "--server.port", "8501",
                  "--server.headless", "true", "--server.enableCORS", "false",
                  "--server.enableXsrfProtection", "false", "--browser.gatherUsageStats", "false"],
                 cwd=ROOT, env=env, stdout=open(ROOT / "streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
cf_log = ROOT / "cloudflared.log"
subprocess.Popen([str(CF), "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                 stdout=open(cf_log, "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", cf_log.read_text())
    if m:
        url = m.group(0); break
print("Dashboard:", url or "tunnel not ready - rerun this cell or check cloudflared.log")
print("(Keep this Colab tab open; the link stops working when the runtime stops. First load may take ~20 s.)")